In [ ]:
!pip install figuard --upgrade -q
print("✓ FiGuard installed")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://sandbox.figuard.io",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   ${budget.total_limit}")
print(f"  Session token: {budget.session_token[:20]}...")

In [ ]:
# Authorized transaction
auth = client.authorize(
    session_token=budget.session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="JetBlue SFO→JFK roundtrip",
    requested_quantity=270.00,
    idempotency_key="booking-001",
)
print(f"Transaction 1: {auth.decision} — ${auth.approved_quantity}")

client.confirm_event(auth.event_id, confirmed_quantity=267.00)
print("✓ Confirmed. Ledger updated.")

# Denied — over remaining budget
auth2 = client.authorize(
    session_token=budget.session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Business class upgrade",
    requested_quantity=890.00,
    idempotency_key="booking-002",
)
print(f"Transaction 2: {auth2.decision} — {auth2.denial_reason}")
print("Nothing moved. Ledger recorded the denial.")

print(f"\n→ See the spend tree: https://sandbox.figuard.io/ui")